In [1]:
import os
from dotenv import load_dotenv
import numpy as np
import time

from trading_functions import (
    initialize_mt5,
    initialize_trading_log,
    load_models,
    load_scaler,
    check_allowed_trading_hours,
    make_order,
    close_all_positions,
    record_trade,
    datetime,
    mt5,
    pd,
    logging,
    OrderExecutor
)

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.FileHandler(r".\bot.log", mode="a"), logging.StreamHandler()],
)

# Get the logger instance
logger = logging.getLogger()
logging.info("Logger initialized")

load_dotenv()
logging.info("Loaded .env file")

LOGIN = os.getenv("LOGIN")
PASSWORD = os.getenv("PASSWORD")
SERVER = os.getenv("SERVER")

symbol = "XAUUSD"
timeframe = mt5.TIMEFRAME_H1
volume = 0.01
strategy_name = "Mentis2026"
sl_price_range = 3
tp_price_range = 3
spread = 0.125
deviation = 0
deviation_delayed_trade = 0.300  # abs(current close price - previous complete close price) for example |1900.000 -1901.111| = 1.111
num_positions_max = 5
magic = 123992

history_name = "trading_log_202609"  # file name for the trading log
history_extension = "csv"  # file extension for the trading log
history_destination = fr".\history\{history_name}.{history_extension}"

# Preparing
logging.info("Preparing")
order_executor = OrderExecutor(
    strategy_name=strategy_name,
    symbol=symbol,
    volume=volume,
    deviation=deviation,
    magic=magic,
    sl_price_range=sl_price_range,
    tp_price_range=tp_price_range,
    spread=spread,
)
initialize_mt5(LOGIN, PASSWORD, SERVER)
trading_log = initialize_trading_log(history_destination, symbol, timeframe)
scaler = load_scaler(r".\models\scaler")
lsvm_long, lsvm_short = load_models(
    r".\models\lsvm_xauusd_long", r".\models\lsvm_xauusd_short"
)

2026-09-29 15:45:57 INFO Logger initialized
2026-09-29 15:45:57 INFO Loaded .env file
2026-09-29 15:45:57 INFO Preparing
2026-09-29 15:45:57 INFO Initilize: True
2026-09-29 15:45:57 INFO Login: False
2026-09-29 15:45:57 INFO | Login: 5056660309, | Balance: 1000.0, | Equity: 1000.0
2026-09-29 15:45:57 INFO Already have a trading log: .\history\trading_log_202609.csv
d:\Git\Trading-bot-lsvm-v2\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.2.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
2026-09-29 15:45:57 INFO Loaded scaler: StandardScaler()
2026-09-29 15:45:57 INFO Loaded models: .\models\lsvm_xauusd_long, .\models\lsvm_xauusd_short


In [15]:
# force Trade
order_result = order_executor.execute_make_order("buy")
if order_result.retcode == mt5.TRADE_RETCODE_DONE:
    ticket = order_result.order
    order_price = order_result[4]

In [27]:
account = mt5.account_info()
print(account.margin_mode)
print("Hedging mode:", mt5.ACCOUNT_MARGIN_MODE_RETAIL_HEDGING)

0
Hedging mode: 2


In [ ]:
from datetime import timedelta
from datetime import datetime

price_data = mt5.copy_rates_from_pos(symbol, timeframe, 0, 2)[0] 
time_trade = datetime.fromtimestamp(price_data[0])
time_trade = time_trade - timedelta(hours=1)
time_trade

datetime.datetime(2026, 9, 29, 16, 0)

In [4]:
price_data = mt5.copy_rates_from_pos(symbol, timeframe, 0, 2)[0] 
current_candle = mt5.copy_rates_from_pos(symbol, timeframe, 0, 2)[1]
open = price_data[1]
high = price_data[2]
low = price_data[3]
close = price_data[4] #This is all bid price on both completed and current candlestick
time_trade = datetime.fromtimestamp(price_data[0])

# Adjust time_trade format
time_trade_str = time_trade.strftime('%Y-%m-%d %H:%M:%S')
time_trade_ts = pd.Timestamp(time_trade_str)

import pandas as pd

file_name = 'trading_log_202609' # file name for the trading log
file_extension = 'csv' # file extension for the trading log
file_destination = f'{file_name}.{file_extension}'

trading_log = pd.read_csv(file_destination)
pandas_time_series = pd.to_datetime(trading_log['time_trade'], format='%Y-%m-%d %H:%M:%S')

current_time_rounded_str = str(time_trade_ts.floor('h'))

history_time_rounded_list = list(pandas_time_series.dt.floor('h'))

history_time_rounded_list_str = [str(i) for i in history_time_rounded_list]



In [5]:
current_time_rounded_str

'2026-09-29 12:00:00'

In [37]:
for i in history_time_rounded_list_str:
    if current_time_rounded_str == i:
        print(i)
        print("fuck you bitche43")
    else:
        print(i)
        print("fuck you boy")

2026-09-29 11:00:00
fuck you boy
2026-09-29 12:00:00
fuck you bitche43


In [6]:
history_time_rounded_list_str

['2026-09-29 11:00:00']